imports

In [15]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import log_loss
from sklearn.model_selection import train_test_split


Train & Test Dataset Split

In [16]:
train = pd.read_csv("../datasets/train.csv")
test = pd.read_csv("../datasets/train.csv")

Overview of the Train Dataset

In [17]:
print("Shape: ", train.shape)
print("Columns: ", train.columns.tolist())

Shape:  (57477, 9)
Columns:  ['id', 'model_a', 'model_b', 'prompt', 'response_a', 'response_b', 'winner_model_a', 'winner_model_b', 'winner_tie']


Feature Engineering

In [18]:
def extract_features(df):
    df['len_a'] = df['response_a'].str.len()
    df['len_b'] = df['response_b'].str.len()
    df['len_diff'] = df['len_a'] - df['len_b']
    df['words_a'] = df['response_a'].str.split().str.len()
    df['words_b'] = df['response_b'].str.split().str.len()
    df['words_diff'] = df['words_a'] - df['words_b']
    df['has_code_a'] = df['response_a'].str.contains('```').astype(int)
    df['has_code_b'] = df['response_b'].str.contains('```').astype(int)
    return df

train = extract_features(train)
test = extract_features(test)

Printing Train Test Dataset After Feature Engineering

In [19]:
print(train)
print(test)

               id             model_a              model_b  \
0           30192  gpt-4-1106-preview           gpt-4-0613   
1           53567           koala-13b           gpt-4-0613   
2           65089  gpt-3.5-turbo-0613       mistral-medium   
3           96401    llama-2-13b-chat  mistral-7b-instruct   
4          198779           koala-13b   gpt-3.5-turbo-0314   
...           ...                 ...                  ...   
57472  4294656694          gpt-4-0613             claude-1   
57473  4294692063          claude-2.0     llama-2-13b-chat   
57474  4294710549            claude-1           alpaca-13b   
57475  4294899228              palm-2       tulu-2-dpo-70b   
57476  4294947231  gemini-pro-dev-api   gpt-4-1106-preview   

                                                  prompt  \
0      ["Is it morally right to try to have a certain...   
1      ["What is the difference between marriage lice...   
2      ["explain function calling. how would you call...   
3      ["How ca

Creating Winner Column

In [20]:
def get_winner(row):
    if row['winner_model_a'] == 1:
        return 'winner_model_a'
    elif row['winner_model_b'] == 1:
        return 'winner_model_b'
    else:
        return 'winner_tie'

train['winner'] = train.apply(get_winner, axis=1)
print(train['winner'].unique())

['winner_model_a' 'winner_model_b' 'winner_tie']


Historical Model Win Rate Features (Day 1)

In [21]:
# Compute win rate per model from training data only
model_a_wins = train[train['winner'] == 'winner_model_a'].groupby('model_a').size()
model_b_wins = train[train['winner'] == 'winner_model_b'].groupby('model_b').size()
total_wins = model_a_wins.add(model_b_wins, fill_value=0)
total_appearances = train.groupby('model_a').size().add(train.groupby('model_b').size(), fill_value=0)
model_winrate = (total_wins / total_appearances).fillna(0.5)

print("Model win rates:")
print(model_winrate.sort_values(ascending=False))

# Map win rates onto train and test
for df in [train, test]:
    df['winrate_a'] = df['model_a'].map(model_winrate).fillna(0.5)
    df['winrate_b'] = df['model_b'].map(model_winrate).fillna(0.5)
    df['winrate_diff'] = df['winrate_a'] - df['winrate_b']

Model win rates:
model_a
gpt-4-1106-preview         0.551374
gpt-3.5-turbo-0314         0.546083
gpt-4-0125-preview         0.513793
gpt-4-0314                 0.483503
claude-1                   0.439165
                             ...   
stablelm-tuned-alpha-7b    0.171206
llama-13b                  0.160878
chatglm3-6b                0.158746
dolly-v2-12b               0.155000
chatglm2-6b                0.129433
Length: 64, dtype: float64


Preparing X and y

In [22]:
features = [
    'len_a', 'len_b', 'len_diff',
    'words_a', 'words_b', 'words_diff',
    'has_code_a', 'has_code_b',
    'winrate_a', 'winrate_b', 'winrate_diff',  # Day 1: historical win rates
]

X_train = train[features]
y_train = train["winner"]

X_test = test[features]

Log Loss Evaluation (Kaggle Metric)

In [ ]:
# Split train into 80% train / 20% validation to measure log loss locally
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

val_model = GradientBoostingClassifier()
val_model.fit(X_tr, y_tr)

val_probs = val_model.predict_proba(X_val)
classes = list(val_model.classes_)

# Build ground-truth probability matrix matching class order
y_val_matrix = pd.get_dummies(y_val)[classes].values

score = log_loss(y_val_matrix, val_probs)
print(f"Validation Log Loss: {score:.4f}")
print("(Lower is better — Kaggle leaderboard uses the same metric)")

Training The Model

In [23]:
model = GradientBoostingClassifier()
model.fit(X_train, y_train)

GradientBoostingClassifier()

In [24]:
print(model.classes_)

['winner_model_a' 'winner_model_b' 'winner_tie']


In [25]:
print(train['winner'].unique())
print(model.classes_)

['winner_model_a' 'winner_model_b' 'winner_tie']
['winner_model_a' 'winner_model_b' 'winner_tie']


In [26]:
probs = model.predict_proba(X_test)
classes = list(model.classes_)

submission = pd.DataFrame({'id': test['id']})
submission['winner_model_a'] = probs[:, classes.index('winner_model_a')]
submission['winner_model_b'] = probs[:, classes.index('winner_model_b')]
submission['winner_tie'] = probs[:, classes.index('winner_tie')]

submission.to_csv('submission.csv', index=False)
print(submission.head())

       id  winner_model_a  winner_model_b  winner_tie
0   30192        0.645284        0.153346    0.201370
1   53567        0.227495        0.473599    0.298906
2   65089        0.287555        0.398704    0.313742
3   96401        0.552374        0.203201    0.244426
4  198779        0.303583        0.419311    0.277106
